# 16 - Task 3: the per-group share surface

**The result.** Thresholding the two test id groups separately, instead of applying one
global share to all 6,999 rows, is worth **+0.022** on the leaderboard: 0.77942 to
**0.80143**. No model was refitted for any of it. Every submission in this notebook is a
different threshold applied to one cached score vector.

**The distinction that made it work.** Global predicted share and per-group predicted
share are different levers, and for six rounds this project only had the first one.
Notebook 09 swept global share and found it flat; notebook 12 swept it again on the
raw-text model and found it flat again (0.4996 gave 0.77349, 0.5299 gave 0.77231, a tie).
Both conclusions were correct and both were misleading, because **a global cut moves both
groups in the same direction and the two groups need corrections in opposite
directions.** The uuid gain was being cancelled by the numeric loss. Separating them
breaks the cancellation.

**The honest frame for the report.** This is a calibration result, not a modelling result.
The classifier is unchanged: LightGBM at default hyperparameters on 40,385 raw-text
features. What changed is where its scores get cut, and roughly a third of this project's
total improvement over the supplied-feature baseline came from that.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import paths, data, ensemble, text, clustering
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. The model being thresholded

Fixed throughout. Nothing here changes it.

| | |
|---|---|
| estimator | `LGBMClassifier(class_weight="balanced", n_jobs=-1, random_state=42)`, **default hyperparameters** |
| fitted on | all 20,000 training rows |
| features | 40,385 columns, blocks A-F and H-I, built from raw text |
| excluded | `G_readability` and the supplied 5,000-dim TF-IDF, both dropped by notebook 14's ablation |
| local Macro F1 | **0.8764** standard 5-fold CV, **0.8089** leave-one-length-band-out |

**The local numbers describe the model, not the submission.** Predicted share cannot be
validated locally at all: dev and holdout are both carved from train, are 100%
uuid-format, and inherit train's 0.6252 class balance, while the test set sits near 0.53.
Everything in this notebook is therefore Kaggle-only evidence, and that limitation is a
property of the task rather than an oversight.

In [ ]:
test_ids, test_texts = text.load_test_text()
groups = text.id_group(test_ids)
masks = text.group_masks(test_ids)

scores = np.load(paths.DATA_PROCESSED / "chosen_test_scores.npy")

# These must be the scores behind the 0.77942 submission or nothing below is on one curve.
ref = pd.read_csv(paths.SUBMISSIONS / "round5_features_share50.csv",
                  dtype={"id": str})["label"].to_numpy()
assert (ensemble.threshold_at_share(scores, 0.4996) == ref).all(), \
    "cached scores do not reproduce round5_features_share50.csv"
print(f"cached scores {scores.shape} reproduce the 0.77942 submission exactly")
print(f"test groups: " + "  ".join(f"{g} {int(m.sum())}" for g, m in masks.items()))

## 2. The measured surface

Every point is the same score vector cut differently. Each row differs from its comparison
point in **one group only**, which is what makes each one readable as an answer to a single
question.

In [ ]:
surface = pd.DataFrame([
    {"file": "round5_features_share50.csv", "uuid": 0.4687, "numeric": 0.5120,
     "kaggle": 0.77942},
    {"file": "chosen_uuid56.csv",           "uuid": 0.5598, "numeric": 0.5120,
     "kaggle": 0.79706},
    {"file": "chosen_uuid62.csv",           "uuid": 0.6198, "numeric": 0.5120,
     "kaggle": 0.80049},
    {"file": "chosen_pergroup62_48.csv",    "uuid": 0.6198, "numeric": 0.4756,
     "kaggle": 0.80143},
    {"file": "chosen_pergroup62_55.csv",    "uuid": 0.6198, "numeric": 0.5484,
     "kaggle": 0.79123},
])
NOISE = 0.0084
surface["vs_best"] = (surface["kaggle"] - surface["kaggle"].max()).round(5)
print(surface.sort_values("kaggle", ascending=False).to_string(index=False))
print(f"\nnoise floor {NOISE}. Anything within that of the best is a tie.")

### 2.1 The uuid axis, numeric held at 0.5120

The slope is how the peak announced itself.

In [ ]:
def axis(df, col, hold_col, hold_val):
    sub = df[np.isclose(df[hold_col], hold_val)].sort_values(col)
    x, yv = sub[col].to_numpy(), sub["kaggle"].to_numpy()
    for (x0, y0), (x1, y1) in zip(zip(x, yv), zip(x[1:], yv[1:])):
        print(f"  {x0:.4f} -> {x1:.4f}   dF1 {y1 - y0:+.5f}   "
              f"slope {(y1 - y0) / (x1 - x0):+.4f}")
    c = np.polyfit(x, yv, 2)
    vx = -c[1] / (2 * c[0])
    return vx, float(np.polyval(c, vx)), x, yv


print("uuid axis (numeric 0.5120):")
vux, vuy, xu, yu = axis(surface, "uuid", "numeric", 0.5120)
print(f"  parabola vertex uuid {vux:.4f}, predicted F1 {vuy:.5f}")

print("\nnumeric axis (uuid 0.6198):")
vnx, vny, xn, yn = axis(surface, "numeric", "uuid", 0.6198)
print(f"  parabola vertex numeric {vnx:.4f}, predicted F1 {vny:.5f}")

print(f"\nshipped point: uuid 0.6198 (vertex {vux:.4f}), "
      f"numeric 0.4756 (vertex {vnx:.4f})")
print(f"remaining headroom on the numeric axis: {vny - 0.80143:+.5f}")
print("Both axes are at their peaks within measurement resolution. Stop searching.")

### What the two axes say

**uuid: slope +0.194, then +0.057.** The first step (0.4687 to 0.5598) gained +0.01764,
the second (to 0.6198) gained +0.00343. A collapse by a factor of 3.4 with the vertex
landing at 0.6214, essentially where the model already sits.

**numeric: an asymmetric peak.** Moving down 0.0364 gained +0.00094; moving up the same
0.0364 lost **-0.00926**. Only the upward step clears the noise floor, and it is what
makes the direction certain rather than assumed. The vertex sits at 0.4897 and the shipped
point is 0.4756, worth +0.0006 more, which is nothing.

**The direction is the interesting part.** uuid wants a *higher* predicted machine share
and numeric wants a *lower* one, which is exactly why every global sweep read as flat. It
also matches the COLING paper's account: the uuid rows share provenance with training data
(62.6% machine) while the numeric rows are the peer-review corpora at a lower rate.

**Testing both directions was worth the slot.** The prediction that numeric should
decrease came from decomposing the flat global sweep, which crossed two representations and
assumed linearity. The upward point cost one submission and converted an inference into a
measurement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (x, yv, vx, vy, name, ship) in zip(axes, [
        (xu, yu, vux, vuy, "uuid rows (numeric held 0.5120)", 0.6198),
        (xn, yn, vnx, vny, "numeric rows (uuid held 0.6198)", 0.4756)]):
    grid = np.linspace(min(x) - 0.02, max(x) + 0.02, 200)
    ax.plot(grid, np.polyval(np.polyfit(x, yv, 2), grid), "-", lw=1, alpha=0.6,
            color="tab:gray")
    ax.plot(x, yv, "o", ms=7)
    ax.axvline(vx, ls="--", lw=0.9, color="tab:red", label=f"vertex {vx:.3f}")
    ax.axvline(ship, ls=":", lw=0.9, color="tab:green", label=f"shipped {ship:.3f}")
    ax.set_xlabel("predicted machine share for this group")
    ax.set_ylabel("Kaggle public Macro F1")
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / "share_surface.png", dpi=120)
plt.show()

## 3. Rebuilding any point on the surface

Kept so the result is reproducible without refitting anything. `threshold_per_group` cuts
each group independently with the exact top-k rule, so a group's realised share lands
within one row of its target regardless of how the two groups' score distributions differ.

In [ ]:
BEST = {"uuid": 0.6198, "numeric": 0.4756}

preds = clustering.threshold_per_group(scores, groups, BEST)
data.write_submission(test_ids, preds, "chosen_pergroup62_48.csv")

sample = pd.read_csv(paths.DATA_RAW / "sample_submission.csv", dtype={"id": str})
sub = pd.read_csv(paths.SUBMISSIONS / "chosen_pergroup62_48.csv", dtype={"id": str})
assert list(sub.columns) == ["id", "label"]
assert list(sub["id"]) == list(sample["id"]), "id order drifted"
assert set(sub["label"].unique()) <= {0, 1}
lab = sub["label"].to_numpy()
for g, m in masks.items():
    assert abs(lab[m].mean() - BEST[g]) < 1 / m.sum(), f"{g} share off target"
print(f"rebuilt the 0.80143 submission: global share {lab.mean():.4f}, "
      + "  ".join(f"{g} {lab[m].mean():.4f}" for g, m in masks.items()))

## 4. The overfitting guard, which now binds

Seven submissions have been spent locating this peak and **every one was scored on the
public leaderboard**, roughly 3,570 rows with a 0.0084 noise floor. A coordinate search
that size can manufacture 0.005 to 0.010 of apparent gain that does not exist on the
private rows.

Read the surface accordingly:

- **0.80143 over 0.80049 (+0.00094) is a tie, not an improvement.** So is 0.80049 over
  0.79706 (+0.00343). Only the first uuid step (+0.01764) and the numeric-up step
  (-0.00926) clear the floor.
- **What is solid** is the *shape*: uuid up, numeric down, both peaking near where the
  model now sits. What is not solid is the last decimal of either coordinate.
- **For the two final picks, do not take the top two public scores.** They differ by 0.0009
  and sit at adjacent points of the same search. Pair the best point with a more central
  one so the pair is not a bet on one noisy optimum.

The probe established that the public set does contain uuid rows, so public feedback does
describe the private population. That justifies trusting the direction. It does not justify
trusting a peak located to within +/- 0.03 in share.

## Discussion / carry-forward -> Task 4 report

**Result: 0.77942 to 0.80143 by thresholding alone, +0.022, no model change.**

### The finding

Global predicted share is flat on this task and per-group predicted share is the largest
single lever in the project. The two look like the same knob and are not. Six rounds
measured the flat one and concluded share was exhausted; that conclusion was correct about
the quantity it measured and wrong about the question it appeared to answer.

The mechanism is worth stating plainly because it generalises: **when a metric is optimised
by one global threshold over a population that is a mixture, the optimum for the mixture
can be flat while the optima for the components are far apart and moving in opposite
directions.** Averaging hid a +0.019 effect and a -0.019 effect.

### Where the total gain came from

| step | Kaggle |
|---|---|
| supplied TF-IDF, LightGBM, global share | 0.73583 |
| raw-text representation, global share | 0.77942 |
| per-group thresholding | **0.80143** |

Roughly two-thirds representation, one-third calibration. For the report that split matters:
the calibration third came from reading the dataset construction, not from any modelling.

### What is closed

- **Global share.** Flat on two different models across 0.44 to 0.56.
- **Per-group share.** Both axes at their vertices within resolution. Further points would
  fit public-leaderboard noise.
- **Feature engineering in the tried directions.** Round 6 tested extended diversity
  (-0.0022 grouped), variability (+0.0049), both (+0.0011) and char n-grams at 5x capacity
  (+0.0023). All inside the bar.

### What remains

1. **Hyperparameter tuning on this representation has never been attempted.** Every search
   in the repo ran against the supplied 5,000 features; this model is LightGBM at
   defaults on 40,385 columns. It is the last untouched axis, and unlike share it can be
   validated locally against the grouped protocol.
2. **The XGBoost blend is unmeasured**, not negative. Notebook 15 section 7 has the cell.
3. **Final pick selection**, per section 4: best point paired with a central one.